# k02-paraphrase-vi
Generated from kaggle/jobs. Requires private input data; no outputs or metrics are fabricated.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", 'transformers==4.46.3', 'datasets>=3,<4', 'accelerate>=1,<2', 'sentence-transformers==3.3.1', 'optimum[onnxruntime]==1.23.3', 'onnx>=1.16,<2', 'onnxruntime>=1.20,<2', 'pyvi>=0.1.1', 'sentencepiece>=0.2', 'protobuf', 'pyarrow', 'pandas', 'scikit-learn', 'scipy'], check=True)


In [ ]:
"""Shared source embedded into each generated standalone Kaggle notebook."""
import os
os.environ['TOKENIZERS_PARALLELISM']='false'
import json
import random
import subprocess
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
print({'torch':torch.__version__,'transformers':transformers.__version__,'seed':SEED})
WORK=Path('/kaggle/working');WORK.mkdir(exist_ok=True)
INPUT=Path('/kaggle/input/ecom-intent-data')
START=time.monotonic()


def input_file(name,required=True):
    candidates=sorted(Path('/kaggle/input').rglob(name))
    if not candidates:
        if required:raise FileNotFoundError(f'Upload {name} to the private ecom-intent-data dataset')
        return None
    if len(candidates)>1:
        raise ValueError(f'Ambiguous input {name}: {candidates}. Attach only one data version.')
    return candidates[0]


def save_json(path,data):
    Path(path).write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False))


def time_budget():
    if time.monotonic()-START>5.5*3600:
        raise RuntimeError('5.5 hour budget reached. Download checkpoint, attach it to the dataset, then resume.')


def normalize_text(text):
    import re,unicodedata
    return re.sub(r'\s+',' ',unicodedata.normalize('NFC',text)).strip()


def preprocess(text):return normalize_text(text).lower()


def segment_vi(text):
    from pyvi import ViTokenizer
    return ViTokenizer.tokenize(preprocess(text))


def export_onnx(source,destination,task):
    """GPU export in Kaggle; local work only consumes the exported ONNX."""
    from optimum.onnxruntime import ORTModelForFeatureExtraction,ORTModelForSequenceClassification,ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig
    cls=ORTModelForFeatureExtraction if task=='feature-extraction' else ORTModelForSequenceClassification
    exported=cls.from_pretrained(str(source),export=True)
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    exported.save_pretrained(destination)
    quantizer=ORTQuantizer.from_pretrained(exported)
    quantizer.quantize(save_dir=destination,quantization_config=AutoQuantizationConfig.avx2(is_static=False,per_channel=False),file_suffix='int8')
    from transformers import AutoTokenizer
    AutoTokenizer.from_pretrained(str(source)).save_pretrained(destination)


In [ ]:
"""k02: two sampled VI translations per training instruction, checkpointed."""
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM


def run_paraphrases():
    if not torch.cuda.is_available():raise RuntimeError('Enable Kaggle GPU')
    inputs=pd.read_parquet(input_file('translation_input.parquet'))
    translated=pd.read_parquet(input_file('bitext_vi_translated.parquet'))
    train=inputs[inputs.split=='train'].sort_values('text_en',key=lambda s:s.str.len())
    checkpoint=WORK/'paraphrase_partial.parquet';prior=input_file('paraphrase_partial.parquet',required=False)
    rows=pd.read_parquet(checkpoint if checkpoint.exists() else prior).to_dict('records') if checkpoint.exists() or prior else []
    train=train[~train.src_id.isin({r['src_id'] for r in rows})]
    name='vinai/vinai-translate-en2vi-v2';tok=AutoTokenizer.from_pretrained(name,src_lang='en_XX');model=AutoModelForSeq2SeqLM.from_pretrained(name).half().cuda().eval()
    target=tok.lang_code_to_id['vi_VN'] if hasattr(tok,'lang_code_to_id') else tok.convert_tokens_to_ids('vi_VN')
    try:
        for offset in range(0,len(train),64):
            time_budget();batch=train.iloc[offset:offset+64]
            encoded=tok(batch.text_en.tolist(),return_tensors='pt',padding=True,truncation=True,max_length=128).to('cuda')
            with torch.inference_mode():out=model.generate(**encoded,decoder_start_token_id=target,do_sample=True,top_p=.9,temperature=.8,num_return_sequences=2,max_new_tokens=128)
            texts=tok.batch_decode(out,skip_special_tokens=True)
            rows.extend({'src_id':sid,'vi_para_1':texts[2*i],'vi_para_2':texts[2*i+1]} for i,sid in enumerate(batch.src_id))
            if offset%4992==0:pd.DataFrame(rows).to_parquet(checkpoint,index=False)
    finally:
        if rows:pd.DataFrame(rows).to_parquet(checkpoint,index=False)
    update=pd.DataFrame(rows).set_index('src_id');translated=translated.set_index('src_id');translated.update(update)
    translated.reset_index().to_parquet(WORK/'bitext_vi_translated.parquet',index=False)


run_paraphrases()
